# Aggregate discrete-choice raw data into instruction-answer pairs

This notebook is for local laptop preprocessing.

In [ ]:
import json
from pathlib import Path
import pandas as pd

In [ ]:
RAW_CSV = Path('raw_mode_choice.csv')
REGION_JSON = Path('regions.json')
OUTPUT_DIR = Path('processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(RAW_CSV)
required_cols = ['ORDER','GROUP','MChoice','MODEAlt','NA1t','TP_order','ORIGIN','DESTIN','TPN','DT','STT','STD','STC']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
print(df.shape)

In [ ]:
MODE_NAME = {'A':'Auto','R':'Riding','S':'Subway','B':'Bus','SB':'Subway&Bus','T':'Taxi','C':'Cycling','W':'Walk'}
TPN_MAP = {1:'Business',2:'Leisure',3:'Commuting',4:'Schooling',5:'Shopping',6:'Other',7:'Returning home'}
DT_MAP = {1:'00:00-06:00',2:'06:00-12:00',3:'12:00-18:00',4:'18:00-24:00'}

def infer_age_group(row):
    if row.get('AGE01',0)==1: return '0-19'
    if row.get('AGE23',0)==1: return '20-39'
    if row.get('AGE45',0)==1: return '40-59'
    if row.get('AGE6',0)==1: return '60+'
    return 'Unknown'

def infer_income_group(row):
    if row.get('INC1',0)==1: return '<1400'
    if row.get('INC2',0)==1: return '1400-3500'
    if row.get('INC3',0)==1: return '>3500'
    return 'Unknown'

In [ ]:
def build_admin_code_map(path):
    if path is None or not Path(path).exists():
        return {}
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    out = {}
    for region in payload.get('regions', []):
        out[str(region.get('adm_cd',''))] = str(region.get('adm_nm',''))
        for district in region.get('districts', []):
            out[str(district.get('adm_cd',''))] = str(district.get('adm_nm',''))
            for town in district.get('towns', []):
                out[str(town.get('adm_cd',''))] = str(town.get('adm_nm',''))
    return out

admin_map = build_admin_code_map(REGION_JSON)

def resolve_place(code):
    s = str(code)
    return admin_map.get(s, f'Unknown(code={s})')

In [ ]:
def make_instruction(profile_row, alternatives):
    gender = 'Male' if int(profile_row.get('MALE',0)) == 1 else 'Female'
    license_text = 'Yes' if int(profile_row.get('LIC',0)) == 1 else 'No'
    purpose = TPN_MAP.get(int(profile_row.get('TPN',6)), 'Other')
    time_window = DT_MAP.get(int(profile_row.get('DT',2)), 'Unknown')

    lines = [
        'You are a transportation planning expert. Given traveler attributes and trip context, predict the most likely travel mode.',
        '',
        'Traveler profile:',
        f'- Gender: {gender}',
        f'- Age group: {infer_age_group(profile_row)}',
        f'- Income level: {infer_income_group(profile_row)}',
        f'- Driver license: {license_text}',
        '',
        'Trip context:',
        f"- Origin: {resolve_place(profile_row.get('ORIGIN'))}",
        f"- Destination: {resolve_place(profile_row.get('DESTIN'))}",
        f'- Trip purpose: {purpose}',
        f'- Departure time window: {time_window}',
        f"- Trip sequence index: {profile_row.get('TP_order',1)}",
        '',
        'Candidate travel modes and attributes:'
    ]

    for alt in alternatives:
        lines.append(f"- {alt['mode_name']}: time={alt['STT']} min, distance={alt['STD']} km, cost={alt['STC']}, available={alt['NA1t']}")

    lines.extend([
        '',
        'Task:',
        'Provide reasoning and then choose the most likely mode.',
        'Output format:',
        'Reasoning: <analysis>',
        'Choice: <one mode name>'
    ])
    return '\n'.join(lines)

In [ ]:
group_keys = ['ORDER','GROUP','TP_order']
records = []

for _, chunk in df.groupby(group_keys, dropna=False):
    chunk = chunk.sort_values('MODEAlt')
    base = chunk.iloc[0].to_dict()
    alternatives = []
    selected_mode = None

    for _, row in chunk.iterrows():
        mode_code = str(row['MODEAlt']).strip()
        mode_name = MODE_NAME.get(mode_code, mode_code)
        alternatives.append({
            'mode_code': mode_code,
            'mode_name': mode_name,
            'NA1t': row.get('NA1t',1),
            'STT': row.get('STT',None),
            'STD': row.get('STD',None),
            'STC': row.get('STC',None),
        })
        if str(row.get('MChoice','0')) in {'1','1.0'}:
            selected_mode = mode_name

    if selected_mode is None:
        continue

    records.append({
        'ORDER': base.get('ORDER'),
        'GROUP': base.get('GROUP'),
        'TP_order': base.get('TP_order'),
        'instruction': make_instruction(base, alternatives),
        'answer': selected_mode,
        'alternatives_json': json.dumps(alternatives, ensure_ascii=False),
    })

agg_df = pd.DataFrame(records)
print('Aggregated samples:', len(agg_df))
agg_df.head(2)

In [ ]:
agg_csv = OUTPUT_DIR / 'aggregated_choice_sets.csv'
agg_json = OUTPUT_DIR / 'instruction_answer.json'
agg_df.to_csv(agg_csv, index=False)
agg_df[['instruction','answer']].to_json(agg_json, orient='records', force_ascii=False, indent=2)
print('Saved:', agg_csv)
print('Saved:', agg_json)

## Laptop feasibility
- CPU-friendly preprocessing, no GPU required.